In [ ]:
import pandas as pd
import requests

url = "https://raw.githubusercontent.com/alura-cursos/challenge2-data-science-LATAM/main/TelecomX_Data.json"
response = requests.get(url)
df = pd.json_normalize(response.json())
df.head()


In [ ]:
df.columns = [col.split('.')[-1] for col in df.columns]
df['Total'] = pd.to_numeric(df['Total'], errors='coerce').fillna(0)
df['Monthly'] = pd.to_numeric(df['Monthly'], errors='coerce')
df.drop_duplicates(inplace=True)

for col in df.columns:
    if df[col].isnull().any():
        if df[col].dtype == 'object':
            df[col].fillna(df[col].mode()[0], inplace=True)
        else:
            df[col].fillna(df[col].median(), inplace=True)

df.to_csv('TelecomX_Cleaned.csv', index=False)
df.info()


In [ ]:
import plotly.express as px

fig = px.pie(df, names='Churn', title='Distribución de Abandono (Churn)')
fig.show()


In [ ]:
fig = px.histogram(df, x='Contract', color='Churn', barmode='group', title='Churn por Tipo de Contrato')
fig.show()


In [ ]:
df['Churn_Num'] = df['Churn'].map({'Yes': 1, 'No': 0})
cols = ['tenure', 'Monthly', 'Total', 'Churn_Num']

df_corr = df[cols].corr()
fig = px.imshow(df_corr, text_auto=".2f", aspect="auto", color_continuous_scale='RdBu_r', title='Matriz de Correlación')
fig.show()


# 📄 Informe Final: Conclusiones sobre la Evasión de Clientes (Churn)

Basado en el Análisis Exploratorio de Datos (EDA) realizado, hemos identificado los siguientes patrones principales que explican el alto índice de evasión:

1. **Tipo de Contrato:** Existe una concentración significativa de abandono en los clientes que poseen **contratos de mes a mes (Month-to-month)**. La falta de un compromiso a largo plazo facilita que estos clientes cancelen el servicio ante cualquier insatisfacción o mejor oferta de la competencia.
2. **Cargos Mensuales:** Se observa una correlación positiva moderada entre los cargos mensuales (`Monthly`) y el Churn. Los clientes que pagan tarifas mensuales más altas tienden a abandonar el servicio con mayor frecuencia, lo que sugiere una posible insatisfacción con el balance precio/valor.
3. **Antigüedad (Tenure):** La matriz de correlación muestra una relación negativa profunda entre la antigüedad del cliente (`tenure`) y el Churn. Los clientes más nuevos son mucho más propensos a irse, mientras que los clientes consolidados son más leales.

**Recomendaciones para el equipo de negocio:**
- Implementar incentivos o descuentos para migrar a los clientes de contratos mensuales a contratos anuales o bianuales.
- Crear programas de retención temprana enfocados específicamente en los primeros meses del ciclo de vida del cliente.
- Revisar la estructura de precios de los planes más costosos para asegurar que el valor percibido se alinee con el costo.